In [1]:
# Parameters
BATCH_MODE = "true"


**Navigation** : [Index](README.md) | [<< Précédent](03b_Typed_Decisions_System1.ipynb) | [Suivant >>](05_RAG_Modern.ipynb)

# Function Calling : Connecter les LLMs au Monde Réel

Dans ce notebook, nous explorons le **Function Calling** (appel de fonctions), qui permet aux modèles d'interagir avec des systèmes externes comme des APIs, bases de données, ou services web.

**Objectifs :**
- Comprendre la structure des Tools dans l'API OpenAI
- Maîtriser tool_choice (auto, required, none)
- Exécuter des appels groupés en un seul tour
- Construire des boucles agentiques

**Prérequis :** Notebook 1 (OpenAI Intro), Notebook 3 (Structured Outputs)

**Durée estimée :** 60 minutes

In [2]:
# Verification des dependances (import guards)
try:
    from openai import OpenAI
    openai_AVAILABLE = True
except ImportError:
    openai_AVAILABLE = False
    print(f'WARNING: openai non installe. Installez avec: pip install openai')

try:
    from dotenv import load_dotenv
    dotenv_AVAILABLE = True
except ImportError:
    dotenv_AVAILABLE = False
    print(f'WARNING: python-dotenv non installe. Installez avec: pip install python-dotenv')


# Installation et configuration
import os
from pathlib import Path
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
%pip install -q openai python-dotenv

import json
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime
import random

# Chargement robuste de la configuration .env
from dotenv import load_dotenv
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")
client = OpenAI()

# Charger le modèle depuis .env ou utiliser gpt-5-mini par défaut
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-mini")
BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() == "true"

print("Client OpenAI initialisé !")
print(f"Modèle par défaut: {DEFAULT_MODEL}")
print(f"Mode batch: {BATCH_MODE}")

Note: you may need to restart the kernel to use updated packages.
.env charge depuis: .env


Client OpenAI initialisé !
Modèle par défaut: gpt-5-mini
Mode batch: False


La configuration de l'environnement est terminée. La prochaine cellule définit les variables globales et initialise le client OpenAI qui sera utilisé dans toutes les démonstrations suivantes.

## 1. Architecture des Tools

Les **Tools** permettent au modèle d'appeler des fonctions définies par l'utilisateur. La structure d'un tool suit le format :

```json
{
  "type": "function",
  "function": {
    "name": "nom_fonction",
    "description": "Description claire pour aider le modèle à choisir",
    "parameters": {
      "type": "object",
      "properties": { ... },  // JSON Schema
      "required": [...]        // Paramètres obligatoires
    }
  }
}
```

**Points clés :**
- La **description** est cruciale : elle guide le modèle pour choisir le bon outil
- Les **parameters** utilisent JSON Schema pour validation
- Le modèle génère les arguments, l'utilisateur exécute la fonction

In [3]:
# Définition d'un outil météo
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Obtenir la météo actuelle d'une ville",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Nom de la ville (ex: Paris, Lyon)"
                },
                "unit": {
                    "type": "string",
                    "enum": ["fahrenheit", "celsius"],
                    "description": "Unité de température"
                }
            },
            "required": ["location"]
        }
    }
}]

print("Tool défini:", tools[0]["function"]["name"])
print("\nStructure du tool:")
print(json.dumps(tools[0], indent=2, ensure_ascii=False))

Tool défini: get_weather

Structure du tool:
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Obtenir la météo actuelle d'une ville",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "Nom de la ville (ex: Paris, Lyon)"
        },
        "unit": {
          "type": "string",
          "enum": [
            "fahrenheit",
            "celsius"
          ],
          "description": "Unité de température"
        }
      },
      "required": [
        "location"
      ]
    }
  }
}


### Interprétation de la structure du tool

L'output affiche la structure complète du tool `get_weather` au format JSON. Observons les éléments clés :

| Champ | Valeur | Rôle |
|-------|--------|------|
| `type` | `"function"` | Indique qu'il s'agit d'un appel de fonction |
| `function.name` | `"get_weather"` | Identifiant unique de la fonction |
| `function.description` | "Obtenir la météo..." | Utilisée par le modèle pour **décider** quand appeler cette fonction |
| `parameters.properties` | `location`, `unit` | Définit les arguments que le modèle doit générer |
| `parameters.required` | `["location"]` | `location` est obligatoire, `unit` est optionnel |

**Point critique** : La **description** est ce qui permet au modèle de choisir le bon outil parmi plusieurs. Plus elle est précise, meilleures seront les décisions du modèle.

> **Analogie** : C'est comme donner une boîte à outils à un assistant - la description de chaque outil (marteau, tournevis) l'aide à choisir le bon pour chaque tâche.

In [4]:
# Implémentation de la fonction météo
def get_weather(location: str, unit: str = "celsius") -> str:
    """Simule une API météo (en production, appeler une vraie API comme OpenWeatherMap)"""
    # Données simulées pour démonstration
    weather_data = {
        "Paris": {"temp_c": 18, "condition": "nuageux", "humidity": 65},
        "Lyon": {"temp_c": 22, "condition": "ensoleillé", "humidity": 45},
        "Marseille": {"temp_c": 26, "condition": "ensoleillé", "humidity": 55},
    }
    
    # Fallback pour villes non répertoriées
    data = weather_data.get(location, {"temp_c": 20, "condition": "variable", "humidity": 50})
    
    # Conversion température
    temp = data["temp_c"] if unit == "celsius" else data["temp_c"] * 9/5 + 32
    unit_symbol = "°C" if unit == "celsius" else "°F"
    
    return json.dumps({
        "location": location,
        "temperature": f"{temp}{unit_symbol}",
        "condition": data["condition"],
        "humidity": f"{data['humidity']}%"
    }, ensure_ascii=False)

# Test de la fonction
print("Test de la fonction get_weather:")
print(get_weather("Paris", "celsius"))
print("\nTest avec Fahrenheit:")
print(get_weather("Lyon", "fahrenheit"))
print("\nTest ville inconnue (fallback):")
print(get_weather("Bordeaux"))

Test de la fonction get_weather:
{"location": "Paris", "temperature": "18°C", "condition": "nuageux", "humidity": "65%"}

Test avec Fahrenheit:
{"location": "Lyon", "temperature": "71.6°F", "condition": "ensoleillé", "humidity": "45%"}

Test ville inconnue (fallback):
{"location": "Bordeaux", "temperature": "20°C", "condition": "variable", "humidity": "50%"}


### Interprétation des tests

**Test 1 - Paris en Celsius :**
Retourne des données simulées pour Paris (18°C, nuageux, 65% humidité). En production, cette fonction appellerait une API réelle comme OpenWeatherMap.

**Test 2 - Lyon en Fahrenheit :**
Conversion automatique : 22°C × 9/5 + 32 = 71.6°F. La fonction gère l'unité de température de manière flexible.

**Test 3 - Bordeaux (fallback) :**
La ville n'existe pas dans `weather_data`, donc la fonction retourne des valeurs par défaut (20°C, variable). Ce mécanisme de **fallback** évite les erreurs et garantit une réponse.

**Points clés :**
- Retour en JSON structuré (facilite le parsing par le modèle)
- Gestion des cas limites (villes inconnues)
- Paramètres optionnels avec valeurs par défaut

> **Production** : Remplacer la simulation par une vraie API et ajouter un cache pour réduire les appels.

## 2. Premier Appel avec Tool

Lorsqu'on passe des `tools` à l'API, le modèle peut décider d'appeler une fonction au lieu de répondre directement. Le paramètre `tool_choice` contrôle ce comportement :

- **`auto`** (défaut) : Le modèle décide s'il doit appeler un outil
- **`required`** : Le modèle DOIT appeler au moins un outil
- **`none`** : Le modèle ne peut PAS appeler d'outils
- **`{"type": "function", "function": {"name": "..."}}`** : Forcer un outil spécifique

In [5]:
# Premier appel avec tool : la question porte sur la météo actuelle, ce que fournit get_weather
messages = [{"role": "user", "content": "Quelle est la météo à Paris aujourd'hui ?"}]

response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto"  # Le modèle décide s'il doit appeler un outil
)

assistant_message = response.choices[0].message
print("Finish reason:", response.choices[0].finish_reason)

# Examiner les tool_calls ou afficher la réponse directe
if assistant_message.tool_calls:
    print("\nLe modèle a décidé d'appeler un outil !")
    print("\nContenu de la réponse du modèle:")
    print(assistant_message.content)
    tool_call = assistant_message.tool_calls[0]
    print("\nTool call détaillé:")
    print(f"  ID: {tool_call.id}")
    print(f"  Fonction: {tool_call.function.name}")
    print(f"  Arguments: {tool_call.function.arguments}")
else:
    print("\nLe modèle a répondu directement sans appeler d'outil.")
    print("\nContenu de la réponse du modèle:")
    print(assistant_message.content)

Finish reason: tool_calls

Le modèle a décidé d'appeler un outil !

Contenu de la réponse du modèle:
None

Tool call détaillé:
  ID: call_gp1mbccuTbWtkltG8xu3W99c
  Fonction: get_weather
  Arguments: {"location":"Paris","unit":"celsius"}


### Interprétation de la réponse

Avec `tool_choice: "auto"`, le modèle **décide lui-même** d'appeler un outil ou de répondre directement. La sortie fraîche de la cellule précédente rend cette décision observable sans la supposer :

- `finish_reason: tool_calls` et une liste `tool_calls` non vide : le modèle demande l'exécution d'un outil ;
- `finish_reason: stop` et aucun `tool_calls` : le modèle répond directement, et son texte est affiché par la branche `else`.

La question porte ici sur la météo actuelle, exactement ce que `get_weather` fournit : le modèle a une raison d'appeler l'outil. Remplacez-la par une demande que l'outil ne couvre pas, par exemple la prévision de demain : le modèle répond alors directement, souvent pour signaler que son outil ne donne que les conditions actuelles.

> **Leçon** : `tool_choice: "auto"` ne **garantit pas** un appel d'outil. Le diagnostic imprimé doit donc dépendre de `message.tool_calls`, jamais d'une hypothèse sur la requête. Pour garantir un appel, utilisez `tool_choice: "required"`.

**Structure d'un tool_call** (lorsque le modèle décide d'appeler un outil) :

| Champ | Valeur | Signification |
|-------|--------|---------------|
| `id` | `call_XXX` | Identifiant unique pour lier la réponse de l'outil |
| `function.name` | `get_weather` | Fonction choisie par le modèle |
| `function.arguments` | `{"location": "...", "unit": "..."}` | Arguments générés (JSON) |

> **Important** : le modèle n'**exécute pas** la fonction lui-même. Il retourne uniquement les instructions (nom + arguments) ; c'est l'application qui exécute la fonction et réinjecte le résultat.

## 3. Flux Complet : Boucle Agentique

Le flux typique d'une conversation avec function calling :

1. **Utilisateur** envoie un message
2. **Modèle** retourne `tool_calls` (ou répond directement si pas besoin d'outils)
3. **Application** exécute les fonctions demandées
4. **Application** injecte les résultats avec `rôle="tool"`
5. **Modèle** utilise les résultats pour formuler une réponse finale
6. Retour à l'étape 2 si le modèle veut appeler d'autres fonctions

Cette boucle est appelée **boucle agentique** car le modèle agit comme un agent autonome qui décide quand et quels outils utiliser.

In [6]:
def run_conversation(user_message: str, tools: list, available_functions: dict):
    """Exécute une conversation complète avec appels de fonctions"""
    messages = [{"role": "user", "content": user_message}]
    
    while True:
        response = client.chat.completions.create(
            model=DEFAULT_MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        assistant_message = response.choices[0].message
        messages.append(assistant_message)
        
        # Si pas d'appel de fonction, on a terminé
        if not assistant_message.tool_calls:
            return assistant_message.content
        
        # Exécuter chaque fonction appelée
        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"  Appel: {function_name}({function_args})")
            
            if function_name in available_functions:
                result = available_functions[function_name](**function_args)
            else:
                result = json.dumps({"error": f"Fonction {function_name} non trouvée"})
            
            # Injecter le résultat dans la conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
    
# Test de la boucle agentique
available_functions = {"get_weather": get_weather}

question = "Quel temps fait-il à Lyon?"
print(f"Question: {question}\n")

result = run_conversation(question, tools, available_functions)
print(f"\nRéponse finale: {result}")

Question: Quel temps fait-il à Lyon?



  Appel: get_weather({'location': 'Lyon', 'unit': 'celsius'})



Réponse finale: À Lyon il fait actuellement 22 °C, ensoleillé, humidité environ 45 %. Voulez-vous la prévision pour les prochaines heures ou les prochains jours ?


### Analyse du flux de la boucle agentique

**Étapes exécutées :**

1. **Tour 1** : Modèle reçoit "Quel temps fait-il à Lyon?"
   - Décision : Appeler `get_weather(location="Lyon")`
   - Retour fonction : `{"location": "Lyon", "temperature": "22°C", "condition": "ensoleillé", ...}`

2. **Tour 2** : Modèle reçoit le résultat de la fonction
   - Décision : Formuler une réponse en langage naturel
   - Pas de `tool_calls` → boucle terminée

**Points clés :**

- Le modèle **décide automatiquement** quand arrêter (pas de `tool_calls` → sortie)
- L'ajout de `rôle="tool"` dans les messages permet au modèle de "voir" les résultats
- Ce pattern est à la base des **agents autonomes** (AutoGPT, BabyAGI, etc.)

> **Note** : Une boucle mal conçue peut tourner indéfiniment. Toujours prévoir une limite `max_iterations`.

## 4. Appels de Fonctions Multiples et Groupés en un seul tour

Le modèle peut appeler **plusieurs fonctions** dans une seule réponse. Ceci est utile pour :
- Répondre à des questions complexes nécessitant plusieurs sources de données
- Exécuter plusieurs actions groupées en un seul tour
- Optimiser le nombre d'aller-retours API

Ajoutons plus de fonctions pour démontrer ce comportement.

In [7]:
# Ajouter plus de fonctions
def get_time(timezone: str = "Europe/Paris") -> str:
    """Retourne l'heure actuelle (simulation)"""
    return json.dumps({
        "timezone": timezone,
        "time": datetime.now().strftime("%H:%M:%S"),
        "date": datetime.now().strftime("%Y-%m-%d")
    })

def create_reminder(title: str, time: str, priority: str = "normal") -> str:
    """Crée un rappel (simulation)"""
    return json.dumps({
        "status": "created",
        "reminder": {"title": title, "time": time, "priority": priority}
    }, ensure_ascii=False)

# Définir les nouveaux tools
extended_tools = tools + [
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "Obtenir l'heure et la date actuelles pour un fuseau horaire donné",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "Fuseau horaire (ex: Europe/Paris, America/New_York)"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "create_reminder",
            "description": "Créer un rappel pour une tâche future",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "Titre du rappel"
                    },
                    "time": {
                        "type": "string",
                        "description": "Heure du rappel (ex: '09:00', 'demain 14h')"
                    },
                    "priority": {
                        "type": "string",
                        "enum": ["low", "normal", "high"],
                        "description": "Priorité du rappel"
                    }
                },
                "required": ["title", "time"]
            }
        }
    }
]

all_functions = {
    "get_weather": get_weather,
    "get_time": get_time,
    "create_reminder": create_reminder
}

print("Fonctions disponibles:", ", ".join(all_functions.keys()))

Fonctions disponibles: get_weather, get_time, create_reminder


### Analyse des nouvelles fonctions

Nous avons ajouté deux nouvelles fonctions pour démontrer les appels multiples :

| Fonction | Objectif | Paramètres | Cas d'usage |
|----------|----------|------------|-------------|
| `get_time()` | Obtenir l'heure actuelle | `timezone` (optionnel) | Planification, rappels temporels |
| `create_reminder()` | Créer un rappel | `title`, `time`, `priority` | Gestion de tâches, productivité |

**Points techniques** :
- Les deux fonctions retournent du JSON structuré (facilite le parsing par le modèle)
- `priority` utilise un `enum` pour contraindre les valeurs possibles (low/normal/high)
- Les paramètres optionnels ont des valeurs par défaut sensibles

Avec ces 3 fonctions (météo, heure, rappel), le modèle peut maintenant orchestrer des scénarios complexes combinant plusieurs sources de données et actions.

In [8]:
# Test avec requête complexe nécessitant plusieurs appels
complex_question = (
    "Quelle heure est-il à Paris et quel temps fait-il? "
    "Crée aussi un rappel pour demain 9h: 'Réunion équipe'"
)

print("Question complexe:")
print(complex_question)
print("\nAppels de fonctions (groupés en un seul tour):")

result = run_conversation(complex_question, extended_tools, all_functions)

print("\nRéponse finale:")
print(result)

Question complexe:
Quelle heure est-il à Paris et quel temps fait-il? Crée aussi un rappel pour demain 9h: 'Réunion équipe'

Appels de fonctions (groupés en un seul tour):


  Appel: get_time({'timezone': 'Europe/Paris'})
  Appel: get_weather({'location': 'Paris', 'unit': 'celsius'})
  Appel: create_reminder({'title': 'Réunion équipe', 'time': 'demain 09:00', 'priority': 'normal'})



Réponse finale:
Il est 04:32:11 (heure locale) le 1er novembre 2026 à Paris.

Météo à Paris : 18°C, nuageux, humidité ~65%.

Rappel créé : "Réunion équipe" pour demain à 09:00 (02/11/2026), priorité normale. Voulez-vous que je modifie l'heure, la priorité ou ajoute une alerte/lieu ?


### Analyse de l'exécution groupée en un seul tour

**Observation clé** : Le modèle a effectué **3 appels de fonction groupés en un seul tour** dans une seule réponse :

1. `get_time(timezone='Europe/Paris')`
2. `get_weather(location='Paris')`
3. `create_reminder(title='Réunion équipe', time='demain 09h')`

**Avantages des appels groupés en un seul tour :**

| Aspect | Sans groupement | Groupées en un seul tour |
|--------|------------------|-------------------|
| **Nombre d'aller-retours API** | 3 tours (1 appel par tour) | 1 tour (3 appels groupés) |
| **Latence totale** | (non mesurée) | (test du flux Python, pas un benchmark fournisseur) |
| **Tokens consommés** | Non mesurés | Non mesurés |

Le notebook démontre uniquement la réduction du nombre de tours LLM pour ce scénario. Il ne compare ni les usages de tokens ni la latence de deux exécutions équivalentes ; aucune économie de tokens ou de temps n'est donc conclue ici.

**Comment le modèle décide-t-il d'exécuter des appels groupés en un seul tour ?**

Le modèle détecte que les 3 fonctions sont **indépendantes** (pas de dépendance de données entre elles). Si une fonction dépend du résultat d'une autre, le modèle demande les appels en plusieurs tours et l'application les exécute dans l'ordre requis.

> **Best practice** : Concevez vos fonctions pour maximiser l'indépendance et permettre le groupement en un seul tour.

### Exemple guidé 1 : Définir et appeler un nouvel outil simple

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20554), intégrée comme exemple guidé.*

Le flux de base des Sections 1 à 3, appliqué à un nouvel outil créé de zéro : un calculateur de pourboire. Le schéma JSON déclare deux paramètres (`amount`, `tip_percent`), la fonction Python rend une chaîne descriptive, et `run_conversation` orchestre l'appel avec une requête utilisateur naturelle.

**Points clés** :
- les deux paramètres sont déclarés **required**, avec des descriptions qui guident le modèle (unités, exemple de valeur) ;
- la fonction ne fait aucun appel d'API : la boucle agentique exécute la fonction puis laisse le modèle reformuler son résultat ;
- la requête (« Ajoute 18 % de pourboire à 42,50 EUR ») fixe les deux arguments, ce qui rend le comportement vérifiable.

In [9]:
# Exemple guide 1 : outil calculate_tip (contribution : un groupe d'etudiants EPF, TP2, PR #20554)
tip_tool = [{
    "type": "function",
    "function": {
        "name": "calculate_tip",
        "description": "Calcule le pourboire a laisser pour un montant donne",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Montant de l'addition en euros"},
                "tip_percent": {"type": "number", "description": "Pourcentage de pourboire (ex. 18 pour 18%)"}
            },
            "required": ["amount", "tip_percent"]
        }
    }
}]

def calculate_tip(amount: float, tip_percent: float) -> str:
    tip = amount * tip_percent / 100
    return f"Pourboire de {tip:.2f} EUR pour une addition de {amount} EUR (total : {amount + tip:.2f} EUR)"

result = run_conversation(
    "Ajoute 18% de pourboire a 42,50 EUR",
    tip_tool,
    {"calculate_tip": calculate_tip},
)
print(result)


  Appel: calculate_tip({'amount': 42.5, 'tip_percent': 18})


18 % de 42,50 EUR = 7,65 EUR. Total avec pourboire : 50,15 EUR.


### Lecture du résultat

Le modèle extrait les deux arguments de la requête — `amount=42.5` et `tip_percent=18` — et appelle `calculate_tip` au premier tour ; la fonction rend le pourboire (7,65 EUR) et le total (50,15 EUR), que le modèle restitue dans sa réponse finale. Le motif est exactement celui de la Section 2 : c'est la **description** du tool qui a permis au modèle de choisir la bonne fonction parmi celles disponibles.

### Exemple guidé 2 : Orchestrateur d'appels groupés en un seul tour

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20554), intégrée comme exemple guidé.*

Un scénario où trois outils indépendants sont mobilisés par une seule requête : recherche d'actualités, conversion de devise et calcul d'itinéraire. Plutôt que de recopier trois schémas à la main, l'exemple construit les tools avec une petite fabrique `make_tool(name, description, properties)` qui remplit le JSON Schema commun — puis inspecte directement la première réponse API.

**Points clés** :
- la fabrique factorise la structure `{"type": "function", "function": {...}}` : seul ce qui distingue un outil (nom, description, propriétés) reste visible ;
- les trois fonctions Python sont simulées et renvoient des chaînes plausibles ;
- l'inspection porte sur `response.choices[0].message.tool_calls` : la question est de savoir si les trois appels arrivent **dans le même tour**, pas de dérouler la boucle.

In [10]:
# Exemple guide 2 : 3 outils appeles en un seul tour (contribution : un groupe d'etudiants EPF, TP2, PR #20554)
def make_tool(name, description, properties):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": list(properties.keys()),
            },
        },
    }

grouped_tools = [
    make_tool("search_news", "Recherche les actualites recentes",
              {"query": {"type": "string"}}),
    make_tool("convert_currency", "Convertit un montant entre deux devises",
              {"amount": {"type": "number"},
               "from_currency": {"type": "string"},
               "to_currency": {"type": "string"}}),
    make_tool("get_route", "Calcule un itineraire",
              {"origin": {"type": "string"}, "destination": {"type": "string"}}),
]

def search_news(query):
    return f"3 articles trouves sur {query}"

def convert_currency(amount, from_currency, to_currency):
    return f"{amount} {from_currency} = {amount * 160:.0f} {to_currency}"

def get_route(origin, destination):
    return f"{origin} -> {destination} : 465 km, environ 4h30"

response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Donne-moi l'actu sur l'IA, convertis 100 EUR en JPY et l'itineraire Paris-Lyon"}],
    tools=grouped_tools,
    tool_choice="auto",
)

tool_calls = response.choices[0].message.tool_calls
print("Nombre d'appels dans le premier tour :", len(tool_calls))
for tc in tool_calls:
    print(tc.function.name, tc.function.arguments)


Nombre d'appels dans le premier tour : 3
search_news {"query": "actualité intelligence artificielle IA France international dernières nouvelles"}
convert_currency {"amount": 100, "from_currency": "EUR", "to_currency": "JPY"}
get_route {"origin": "Paris", "destination": "Lyon"}


### Lecture du résultat

La première réponse contient les trois appels d'un coup — `search_news`, `convert_currency`, `get_route` — chacun avec ses arguments extraits de la requête. C'est le comportement « groupé en un seul tour » de la Section 4 : le modèle parallélise les sources indépendantes au lieu de les chaîner, et une seule requête API mobilise les trois outils.

## 5. Contrôle Avancé : tool_choice

Le paramètre `tool_choice` offre un contrôle fin sur le comportement du modèle :

| Valeur | Comportement |
|--------|-------------|
| `"auto"` | Le modèle décide (défaut) |
| `"required"` | Le modèle DOIT appeler au moins un outil |
| `"none"` | Le modèle ne peut PAS appeler d'outils |
| `{"type": "function", "function": {"name": "X"}}` | Force l'appel de la fonction X |

**Cas d'usage :**
- `"required"` : Forcer l'utilisation d'outils pour des données en temps réel
- `"none"` : Désactiver temporairement les outils (mode conversation pure)
- Spécifique : Garantir qu'une action précise sera exécutée

In [11]:
# Test 1: Forcer l'appel d'une fonction spécifique
print("=== Test 1: Forcer l'appel d'un outil spécifique ===")
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Bonjour, comment vas-tu?"}],
    tools=tools,
    tool_choice={"type": "function", "function": {"name": "get_weather"}}
)

print("Question: Bonjour, comment vas-tu?")
print("\nAvec tool_choice forcé à get_weather:")
if response.choices[0].message.tool_calls:
    tc = response.choices[0].message.tool_calls[0]
    print(f"  Le modèle a appelé: {tc.function.name}")
    print(f"  Arguments: {tc.function.arguments}")

# Test 2: Désactiver les tools
print("\n=== Test 2: Désactiver les tools ===")
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Quelle est la météo à Paris?"}],
    tools=tools,
    tool_choice="none"
)

print("Question: Quelle est la météo à Paris?")
print("\nAvec tool_choice='none' (pas d'appel de fonction):")
print(f"  Réponse directe: {response.choices[0].message.content[:100]}...")

# Test 3: Forcer au moins un appel (required)
print("\n=== Test 3: tool_choice='required' ===")
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Dis-moi bonjour"}],
    tools=extended_tools,
    tool_choice="required"
)

print("Question: Dis-moi bonjour")
print("\nAvec tool_choice='required':")

if response.choices[0].message.tool_calls:
    contenu_reponse = response.choices[0].message.content
    print("\nContenu de la réponse du modèle:")
    print(contenu_reponse)
    tc = response.choices[0].message.tool_calls[0]
    print(f"  Le modèle est FORCÉ d'appeler un outil: {tc.function.name} avec les paramètres {tc.function.arguments}")

=== Test 1: Forcer l'appel d'un outil spécifique ===


Question: Bonjour, comment vas-tu?

Avec tool_choice forcé à get_weather:
  Le modèle a appelé: get_weather
  Arguments: {"location":"Paris","unit":"celsius"}

=== Test 2: Désactiver les tools ===


Question: Quelle est la météo à Paris?

Avec tool_choice='none' (pas d'appel de fonction):
  Réponse directe: Je n’ai pas accès aux données météo en temps réel en ce moment, donc je ne peux pas te donner la sit...

=== Test 3: tool_choice='required' ===


Question: Dis-moi bonjour

Avec tool_choice='required':

Contenu de la réponse du modèle:
None
  Le modèle est FORCÉ d'appeler un outil: get_time avec les paramètres {"timezone":"Europe/Paris"}


### Analyse des comportements de tool_choice

| Mode | Requête | Résultat | Observation |
|------|---------|----------|-------------|
| **Forcé spécifique** | "Bonjour, comment vas-tu?" | Appel forcé à `get_weather` | Le modèle DOIT appeler la fonction même si inappropriée |
| **none** | "Quelle est la météo à Paris?" | Réponse textuelle sans fonction | Le modèle répond directement sans accès aux données réelles |
| **required** | "Dis-moi bonjour" | Appel forcé à un outil (ex: `get_time`) | Le modèle choisit l'outil le moins inapproprié |

**Implications pratiques :**
- `tool_choice` forcé peut générer des appels non pertinents → utiliser avec parcimonie
- `tool_choice="none"` utile pour économiser des appels API coûteux quand les données ne changent pas
- `tool_choice="required"` garantit l'utilisation de données temps réel (ex: prix en bourse)

> **Attention** : Forcer un outil peut dégrader l'expérience utilisateur si mal utilisé.

## 6. Gestion Robuste des Erreurs

Dans un système de production, la gestion d'erreurs est cruciale :

**Sources d'erreurs possibles :**
1. **Erreurs API** : Timeout, limites de taux, problèmes réseau
2. **Arguments invalides** : JSON malformé, types incorrects
3. **Fonction non implémentée** : Le modèle appelle une fonction déclarée que l'application n'exécute pas
4. **Erreur d'exécution** : La fonction échoue (ex: API externe indisponible)
5. **Boucles infinies** : Le modèle continue d'appeler des fonctions sans fin

**Bonnes pratiques :**
- Limiter le nombre d'itérations (max_iterations)
- Valider les arguments JSON
- Retourner des messages d'erreur structurés au modèle
- Logger les erreurs pour débogage

In [12]:
def run_safe_conversation(user_message: str, tools: list, available_functions: dict, max_iterations: int = 5):
    """Version robuste avec gestion d'erreurs et limite d'itérations"""
    messages = [{"role": "user", "content": user_message}]

    for iteration in range(max_iterations):
        # Gestion erreurs API
        try:
            response = client.chat.completions.create(
                model=DEFAULT_MODEL,
                messages=messages,
                tools=tools,
                tool_choice="auto"
            )
        except Exception as e:
            return f"Erreur API: {e}"

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        if not assistant_message.tool_calls:
            return assistant_message.content

        # Traiter chaque appel de fonction
        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name

            # Gestion erreurs JSON
            try:
                function_args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError:
                result = json.dumps({"error": "Arguments invalides, JSON malformé"}, ensure_ascii=False)
                print(f"  Appel: {function_name}(<JSON malformé>)\n    -> {result}")
                messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
                continue
            print(f"  Appel: {function_name}({function_args})")

            # Vérifier que la fonction existe
            if function_name not in available_functions:
                result = json.dumps({"error": f"Fonction '{function_name}' non implémentée"}, ensure_ascii=False)
                print(f"    -> {result}")
            else:
                # Gestion erreurs d'exécution
                try:
                    result = available_functions[function_name](**function_args)
                except Exception as e:
                    result = json.dumps({"error": f"Erreur d'exécution: {str(e)}"}, ensure_ascii=False)
                    print(f"    -> {result}")

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return f"Limite d'itérations atteinte ({max_iterations}). Conversation trop longue."

# Test de gestion d'erreurs
print("=== Test 1: Ville non répertoriée (fallback) ===")
print(run_safe_conversation("Météo à Berlin?", tools, all_functions))

print("\n=== Test 2: Outil déclaré mais non implémenté ===")
# get_air_quality est déclaré au modèle, mais absent de all_functions : la garde doit intercepter l'appel
air_quality_tool = {
    "type": "function",
    "function": {
        "name": "get_air_quality",
        "description": "Obtenir l'indice de qualité de l'air actuel d'une ville",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Nom de la ville (ex: Paris, Lyon)"}
            },
            "required": ["city"]
        }
    }
}
print(run_safe_conversation("Quelle est la qualité de l'air à Paris en ce moment ?", extended_tools + [air_quality_tool], all_functions))

print("\n=== Test 3: Requête inexploitable ===")
print(run_safe_conversation("Météo à ????? température en @@@@", tools, all_functions))

=== Test 1: Ville non répertoriée (fallback) ===


  Appel: get_weather({'location': 'Berlin', 'unit': 'celsius'})


Il fait 20 °C à Berlin, temps variable, humidité 50 %. Souhaitez-vous la prévision pour les prochaines heures ou plusieurs jours ?

=== Test 2: Outil déclaré mais non implémenté ===


  Appel: get_air_quality({'city': 'Paris'})
    -> {"error": "Fonction 'get_air_quality' non implémentée"}


Désolé — je n’ai pas pu récupérer la qualité de l’air en temps réel (erreur côté service). Je peux toutefois :

- Réessayer maintenant si vous le souhaitez.  
- Vous indiquer où consulter la valeur immédiatement (sites et applis fiables).  
- Donner une explication des indices AQI et des conseils santé/mesures à prendre selon les niveaux.

Pour information rapide, voici où regarder tout de suite :
- Airparif (pour l’Île‑de‑France) : airparif.fr  
- World Air Quality / WAQI : waqi.info (recherche « Paris »)  
- IQAir / AirVisual : iqair.com  
- Applis : AirVisual (IQAir), BreezoMeter, Plume Labs Now

Et un rappel utile sur l’échelle AQI (approximative) et conseils :
- 0–50 (Bon) : pas de restriction.  
- 51–100 (Modéré) : personnes très sensibles peuvent ressentir des symptômes légers.  
- 101–150 (Mauvais pour certains) : limiter efforts prolongés à l’extérieur pour personnes sensibles (enfants, personnes âgées, asthmatiques).  
- 151–200 (Mauvais) : réduire les activités physiques ext

Vous voulez la météo actuelle, mais il manque la ville et l’unité. Remplacez les placeholders et je lance la recherche.

Exemples valides :
- Météo à Paris température en celsius
- Météo à New York température en fahrenheit
- Météo à Lyon température en C

Souhaitez-vous la météo actuelle uniquement, ou aussi vent, humidité et prévisions ? Donnez la ville et l’unité (celsius/fahrenheit) et je récupère l’info.


### Analyse des résultats de gestion d'erreurs

`run_safe_conversation()` affiche chaque appel d'outil (`Appel:`) et chaque erreur renvoyée au modèle (`->`). Sans cette trace, une réponse polie du modèle ne dit pas si l'outil a été appelé, ni si une garde a servi.

| Test | Scénario | Ce que montre la sortie |
|------|----------|-------------------------|
| **Ville non répertoriée** | Berlin est absent de `weather_data` | l'appel `get_weather` aboutit, avec les valeurs par défaut de la fonction (20°C, variable) |
| **Outil non implémenté** | `get_air_quality` est déclaré au modèle, mais absent de `all_functions` | le modèle appelle `get_air_quality` ; la garde renvoie `{"error": "Fonction 'get_air_quality' non implémentée"}` au lieu de lever une `KeyError`, et le modèle l'annonce à l'utilisateur |
| **Requête inexploitable** | caractères spéciaux (`?????`, `@@@@`) | le modèle n'invente pas d'arguments : il n'appelle aucun outil et demande une précision |

**Points clés :**
- Un modèle n'appelle en pratique que les outils qu'on lui déclare. Une fonction « non implémentée » vient donc d'un écart entre les schémas déclarés et les fonctions implémentées, par exemple un outil annoncé avant d'être branché : c'est ce que reproduit le test 2. Il porte sur une consultation, sans effet de bord : face à une action engageante (réserver, payer), le modèle demande souvent confirmation avant d'appeler l'outil, et la garde ne serait pas atteinte.
- La branche « JSON malformé » reste une garde. Les modèles récents produisent rarement des arguments non parsables, et aucun de ces trois essais ne la déclenche.
- Chaque erreur est renvoyée au modèle sous forme structurée (`role="tool"`), ce qui lui permet d'adapter sa réponse au lieu d'interrompre la conversation.

> **Production** : En environnement réel, logguer toutes les erreurs pour analyse et amélioration continue.

### Exemple guidé 3 : Analyseur de logs avec tool_choice

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20554), intégrée comme exemple guidé.*

Un outil `analyze_log` qui classe un message de log (ERROR, WARNING, INFO) à partir de mots-clés, confronté aux modes de `tool_choice` de la Section 5. Une petite fonction `tester(question, mode)` rejoue la même configuration API en changeant uniquement `tool_choice`, pour observer quand le modèle appelle l'outil — et quand il y est forcé alors qu'il n'a rien à analyser.

**Points clés** :
- la fonction Python rend un JSON structuré (`severity`, `summary`), comme toutes les fonctions outillées de ce notebook ;
- quatre combinaisons sont testées : un log d'erreur avec `auto`, une question sans rapport avec `auto` puis `required`, et un log d'erreur avec `none` ;
- le contraste `required`/`none` sur des questions sans rapport montre ce que le modèle fait quand on lui retire ou impose le choix.

In [13]:
# Exemple guide 3 : analyseur de logs et tool_choice (contribution : un groupe d'etudiants EPF, TP2, PR #20554)
log_tool = [{
    "type": "function",
    "function": {
        "name": "analyze_log",
        "description": "Detecte le niveau de severite (ERROR, WARNING, INFO) d'un message de log",
        "parameters": {
            "type": "object",
            "properties": {
                "log_message": {"type": "string", "description": "Le message de log a analyser"}
            },
            "required": ["log_message"]
        }
    }
}]

def analyze_log(log_message: str) -> str:
    texte = log_message.lower()
    if "error" in texte or "erreur" in texte or "failed" in texte:
        severity = "ERROR"
    elif "warn" in texte or "attention" in texte:
        severity = "WARNING"
    else:
        severity = "INFO"
    return json.dumps({"severity": severity, "summary": log_message[:50]})

def tester(question, mode):
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{"role": "user", "content": question}],
        tools=log_tool,
        tool_choice=mode,
    )
    message = response.choices[0].message
    if message.tool_calls:
        args = json.loads(message.tool_calls[0].function.arguments)
        print(f"[{mode}] outil appele ->", analyze_log(**args))
    else:
        print(f"[{mode}] pas d'outil ->", message.content)

log_erreur = "Analyse ce log : 2024-05-01 ERROR Connection failed to database"
tester(log_erreur, "auto")
tester("Quelle est la capitale de la France ?", "auto")
tester("Quelle est la capitale de la France ?", "required")
tester(log_erreur, "none")


[auto] outil appele -> {"severity": "ERROR", "summary": "2024-05-01 ERROR Connection failed to database"}


[auto] pas d'outil -> La capitale de la France est Paris.


[required] outil appele -> {"severity": "INFO", "summary": "Question utilisateur: \"Quelle est la capitale de l"}


[none] pas d'outil -> Résumé rapide
- Gravité : ERROR — priorité élevée. Indique qu’une opération prévue a échoué (connexion à la base de données).
- Contexte minimal fourni : date (2024-05-01) et message « Connection failed to database ». Pas d’informations sur l’application, l’instance DB, l’hôte/port, ou le message d’erreur détaillé.

Causes probables (ordre fréquent)
1. Service base de données arrêté ou planté.
2. Mauvais host/port ou résolution DNS incorrecte.
3. Problème réseau / pare‑feu / règles de sécurité (ports bloqués).
4. Identifiants (user/password) invalides ou expirés.
5. Limite de connexions atteinte côté serveur.
6. Problème TLS/SSL (certificat expiré, négociation échouée).
7. Problème de configuration côté application (chaîne de connexion mal formée).
8. Ressources serveur saturées (CPU, RAM, I/O) empêchant nouvelles connexions.
9. Changements récents (déploiement, migration, rotation de secrets).
10. Problème côté client (bibliothèque DB obsolète/incompatible).

Pre

### Lecture du résultat

Avec `auto`, le modèle appelle l'outil pour le log d'erreur (classé ERROR par la fonction) et y renonce pour la question sur la capitale de la France. Avec `required`, il est **contraint** d'appeler l'outil alors qu'il n'a aucun log à analyser : les arguments passés sont vides ou génériques, et la fonction les classe INFO — un appel techniquement correct mais sans valeur. Avec `none`, le modèle rédige une analyse libre du log, sans la structure `severity`/`summary` que seul l'outil garantit. Le mode `tool_choice` ne dit pas seulement *si* l'outil est appelé : il change la *nature* du résultat obtenu.

## 7. Cas d'Usage Avancés

Voyons quelques patterns avancés de function calling :

### A. Recherche de base de données
Simuler une recherche dans une base de données avec filtres.

In [14]:
# Simuler une base de données de cours
COURSE_DB = [
    {"id": 1, "title": "Python pour débutants", "category": "Programming", "duration_hours": 10, "level": "Débutant", "price": 49},
    {"id": 2, "title": "Machine Learning Intro", "category": "ML", "duration_hours": 15, "level": "Intermédiaire", "price": 99},
    {"id": 3, "title": "Deep Learning Avancé", "category": "ML", "duration_hours": 30, "level": "Avancé", "price": 299},
    {"id": 4, "title": "Data Science avec R", "category": "Data", "duration_hours": 20, "level": "Intermédiaire", "price": 149},
    {"id": 5, "title": "Computer Vision", "category": "ML", "duration_hours": 25, "level": "Avancé", "price": 349},
]

def search_courses(category: str = None, min_duration: int = None, max_price: int = None, level: str = None) -> str:
    """Recherche de cours avec filtres multiples"""
    results = COURSE_DB.copy()
    
    if category:
        results = [c for c in results if c["category"] == category]
    if min_duration:
        results = [c for c in results if c["duration_hours"] >= min_duration]
    if max_price:
        results = [c for c in results if c["price"] <= max_price]
    if level:
        results = [c for c in results if c["level"] == level]
    
    return json.dumps({"count": len(results), "courses": results}, ensure_ascii=False)

# Définir le tool
search_tool = [{
    "type": "function",
    "function": {
        "name": "search_courses",
        "description": "Rechercher des cours dans la base de données avec filtres optionnels",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {"type": "string", "enum": ["Programming", "ML", "Data"], "description": "Catégorie de cours"},
                "min_duration": {"type": "integer", "description": "Durée minimale en heures"},
                "max_price": {"type": "integer", "description": "Prix maximum en euros"},
                "level": {"type": "string", "enum": ["Débutant", "Intermédiaire", "Avancé"], "description": "Niveau"}
            }
        }
    }
}]

search_functions = {"search_courses": search_courses}

# Test
question = "Trouve-moi tous les cours de Machine Learning de plus de 20 heures"
print(f"Question: {question}\n")
print(run_safe_conversation(question, search_tool, search_functions))

Question: Trouve-moi tous les cours de Machine Learning de plus de 20 heures



  Appel: search_courses({'category': 'ML', 'min_duration': 21})


J’ai trouvé 2 cours de Machine Learning de plus de 20 heures :

1) Deep Learning Avancé  
   - Durée : 30 heures  
   - Niveau : Avancé  
   - Prix : 299 €  
   - ID : 3

2) Computer Vision  
   - Durée : 25 heures  
   - Niveau : Avancé  
   - Prix : 349 €  
   - ID : 5

Souhaitez-vous que je :
- affiche le plan détaillé d’un de ces cours (précisez l’ID ou le titre) ?
- compare les deux (contenu, prérequis) ?
- filtre selon un prix maximum ou un niveau différent ?
- vous inscrive à l’un d’eux ?

Dites-moi ce que vous préférez.


### Interprétation des résultats de recherche

**Requête analysée par le modèle :**
- Catégorie : Machine Learning
- Durée minimale : 20 heures

**Résultats retournés :**
Le modèle a correctement extrait les critères et construit les arguments de la fonction `search_courses()`. Notez que :

1. **Traitement du langage naturel** : "de plus de 20 heures" → `min_duration=20`
2. **Mapping de catégorie** : "Machine Learning" → `category="ML"`
3. **Filtres combinés** : Les deux critères sont appliqués en ET logique

Ce pattern de recherche s'applique à de nombreux cas d'usage réels (e-commerce, CRM, documentation, etc.).

### B. Orchestration Multi-Étapes

Le modèle peut orchestrer plusieurs étapes complexes automatiquement.

In [15]:
# Le modèle peut décomposer une tâche complexe en plusieurs appels
# Demande actionnable : chaque élément correspond à un outil (create_reminder, get_weather)
scenario = ("Crée trois rappels pour demain : réveil à 7h, sport à 8h, travail à 9h. "
            "Dis-moi aussi la météo à Paris, pour savoir si je fais mon sport dehors.")
print(f"Scénario: {scenario}\n")
print("Étapes exécutées:")

result = run_conversation(scenario, extended_tools, all_functions)

print("\nRésultat:")
print(result)

Scénario: Crée trois rappels pour demain : réveil à 7h, sport à 8h, travail à 9h. Dis-moi aussi la météo à Paris, pour savoir si je fais mon sport dehors.

Étapes exécutées:


  Appel: create_reminder({'title': 'Réveil', 'time': 'demain 07:00', 'priority': 'normal'})
  Appel: create_reminder({'title': 'Sport', 'time': 'demain 08:00', 'priority': 'normal'})
  Appel: create_reminder({'title': 'Travail', 'time': 'demain 09:00', 'priority': 'normal'})
  Appel: get_weather({'location': 'Paris', 'unit': 'celsius'})



Résultat:
C’est fait — j’ai créé les trois rappels pour demain :
- Réveil : demain 07:00
- Sport : demain 08:00
- Travail : demain 09:00

Météo à Paris pour demain (résumé) :
- Température : 18°C
- Condition : nuageux
- Humidité : 65%

Conseil rapide : 18°C et nuageux, c’est plutôt confortable pour faire du sport dehors. Il n’y a pas d’indication de pluie dans ces données, mais si tu veux être sûr, je peux vérifier la probabilité de pluie horaire ou te donner une mise à jour plus proche de l’heure. Veux‑tu que je fasse ça ou que je modifie/ajoute quelque chose aux rappels ?


### Interprétation de l'orchestration

Que lire dans la sortie :

- **Les lignes `Appel:`**, une par action. La demande en contient quatre : trois rappels (`create_reminder`) et la météo (`get_weather`). Comptez les appels, puis vérifiez les arguments extraits de la phrase : titre et heure de chaque rappel, ville de la météo.
- **Le regroupement.** Le modèle peut émettre plusieurs appels dans un même tour, comme à la section 4, ou les enchaîner sur plusieurs tours. `run_conversation` exécute les deux formes ; seul le nombre d'allers-retours avec l'API change.
- **La réponse finale.** Elle doit reprendre les résultats des outils (statut `created` des rappels, météo simulée de Paris) et en tirer la conclusion demandée sur le sport en extérieur.

> **Demande actionnable ou intention ouverte.** Remplacez la demande par « Planifie ma journée demain - réveil à 7h, sport à 8h, travail à 9h » et ré-exécutez : face à une intention ouverte, le modèle propose souvent un planning en texte et demande confirmation avant de créer des rappels, sans appeler d'outil. Une demande qui nomme les actions lui laisse moins de place pour cette prudence. Pour imposer au moins un appel, `tool_choice: "required"` reste l'outil (section 5).

### Exemple guidé 4 : Outil de conversion de devises

*Contribution d'un groupe d'étudiants EPF (TP2, PR #20554), intégrée comme exemple guidé.*

Un outil `convert_currency` intégré à la boucle robuste de la Section 6 : la fonction applique des taux fixes avec rejet des devises inconnues (JSON d'erreur structuré), et la question test demande **deux conversions à la fois** — le modèle doit appeler l'outil pour chaque devise.

**Points clés** :
- les taux sont codés dans un dictionnaire `rates` : la fonction renvoie le montant converti arrondi, ou `{"error": "Devise inconnue"}` si une devise est absente ;
- l'outil est ajouté à `extended_tools` et à `all_functions`, puis la question passe par `run_safe_conversation`, qui gère erreurs d'API, arguments invalides et fonctions inconnues ;
- la question (« Combien font 100 euros en dollars et en yen ? ») mobilise la boucle agentique complète : appels multiples, réinjection des résultats, réponse de synthèse.

In [16]:
# Exemple guide 4 : outil de conversion de devises (contribution : un groupe d'etudiants EPF, TP2, PR #20554)
currency_tool = [{
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description": "Convertit un montant d'une devise vers une autre",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Montant a convertir"},
                "from_currency": {"type": "string", "description": "Devise source (EUR, USD, JPY, GBP)"},
                "to_currency": {"type": "string", "description": "Devise cible (EUR, USD, JPY, GBP)"}
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    }
}]

def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    rates = {"EUR": 1.0, "USD": 1.10, "JPY": 160.0, "GBP": 0.85}
    if from_currency not in rates or to_currency not in rates:
        return json.dumps({"error": "Devise inconnue"})
    result = amount / rates[from_currency] * rates[to_currency]
    return json.dumps({"amount": amount, "from": from_currency, "to": to_currency, "result": round(result, 2)})

all_tools_with_currency = extended_tools + currency_tool
all_functions["convert_currency"] = convert_currency
result = run_safe_conversation(
    "Combien font 100 euros en dollars et en yen ?",
    all_tools_with_currency,
    all_functions
)
print(result)


  Appel: convert_currency({'amount': 100, 'from_currency': 'EUR', 'to_currency': 'USD'})
  Appel: convert_currency({'amount': 100, 'from_currency': 'EUR', 'to_currency': 'JPY'})


100 € ≈ 110,00 $ (soit 1 € ≈ 1,10 $)
100 € ≈ 16 000 ¥ (soit 1 € ≈ 160 ¥)

Ces valeurs sont des conversions indicatives (taux = 1,10 USD/€ et 160 JPY/€). Les taux de change varient et les frais bancaires peuvent s’appliquer — voulez-vous que je vérifie le taux exact actuel ou que je convertisse un autre montant ?


### Lecture du résultat

Les lignes `Appel:` montrent les conversions demandées par le modèle, une par devise, avant la réponse de synthèse. Vérifiez que les montants de la réponse sont ceux que renvoie la fonction avec ses taux fixes : 110,00 USD et 16 000 JPY pour 100 EUR. Un modèle qui répondrait sans appeler l'outil donnerait des montants tirés de ses connaissances, sans garantie qu'ils correspondent à ces taux.

Le chemin de robustesse reste disponible : si le modèle proposait une devise absente du dictionnaire `rates`, la fonction renverrait `{"error": "Devise inconnue"}`, et la boucle continuerait sur cette erreur plutôt que de s'interrompre.

## Exercices à compléter

Les quatre exercices suivants mesurent des comportements qu'**aucun exemple guidé de ce notebook ne couvre** : un schéma à paramètres imbriqués, la mémoire des résultats d'outils entre les tours, le surcoût de tokens des schémas déclarés, et l'ancrage d'une réponse contre l'hallucination. Ils réutilisent le `client`, `run_conversation`, `run_safe_conversation`, `extended_tools`, `get_weather` et `COURSE_DB` définis plus haut.

### Exercice 1 : Paramètres imbriqués — un outil qui reçoit une liste d'objets

Tous les outils de ce notebook reçoivent des paramètres **scalaires** (chaîne, nombre). Un outil de planification réaliste reçoit une **liste d'étapes**, chacune étant un objet structuré : le JSON Schema du paramètre devient un `array` dont les `items` sont eux-mêmes un `object`.

**Objectif** : définir un outil `plan_revision` dont le paramètre `matieres` est un tableau d'objets `{"matiere": string, "duree_heures": number}`, implémenter la fonction Python qui **valide chaque élément** (matière non vide, durée strictement positive) et renvoie le total d'heures, puis passer la demande « Planifie mes révisions : maths 2 heures, histoire 1,5 heure, anglais 1 heure. » en un seul appel API et exécuter la fonction sur les arguments renvoyés par le modèle.

**Indices** :
- `# Indice` : dans le schéma, `"matieres": {"type": "array", "items": {"type": "object", "properties": {...}, "required": ["matiere", "duree_heures"]}}`
- `# Indice` : la fonction Python reçoit `matieres` déjà désérialisée en liste de dictionnaires — valider chaque élément dans une boucle
- `# Indice` : en cas d'élément invalide, renvoyer un JSON d'erreur structuré (comme `convert_currency` le fait pour une devise inconnue), pas une exception

In [17]:
# Exercice 1 : Parametres imbriques -- un outil qui recoit une liste d'objets
# TODO etudiant : definir l'outil plan_revision (parametre tableau d'objets) et l'appeler

# Etape 1 : Definir le schema du tool avec un parametre "matieres" tableau d'objets
revision_tool = None  # TODO etudiant : [{"type": "function", "function": {"name": "plan_revision",
#   "parameters": {"properties": {"matieres": {"type": "array", "items": {"type": "object",
#   "properties": {"matiere": {"type": "string"}, "duree_heures": {"type": "number"}},
#   "required": ["matiere", "duree_heures"]}}}, "required": ["matieres"]}}}]

# Etape 2 : Implementer la fonction Python qui valide chaque element
def plan_revision(matieres):
    """Valide chaque matiere (nom non vide, duree > 0) et rend le total d'heures en JSON."""
    # TODO etudiant : boucler sur les elements ; JSON d'erreur structure si un element est invalide
    return None  # TODO etudiant : json.dumps du total d'heures en cas de succes

# Etape 3 : Un appel API unique, puis executer la fonction sur les arguments renvoyes
demande_revision = "Planifie mes revisions : maths 2 heures, histoire 1,5 heure, anglais 1 heure."
# TODO etudiant : client.chat.completions.create(..., tools=revision_tool), extraire les
#   arguments de la reponse et appeler plan_revision(**args)

total_heures = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


### Exercice 2 : Conversation multi-tours — réutiliser un résultat d'outil au tour suivant

`run_conversation` repart à chaque appel d'une liste `messages` vide : les résultats d'outils d'une question sont perdus pour la suivante. Un assistant réel conserve son **historique** — y compris les messages `tool` — d'un tour à l'autre.

**Objectif** : construire manuellement un historique qui survit aux trois tours. Tour 1 : « Quel temps fait-il à Lyon ? » — laisser le modèle appeler `get_weather`, exécuter la fonction, réinjecter le résultat avec le bon `tool_call_id`. Tour 2 : « Et à Marseille ? » — même historique, même mécanique. Tour 3 : « Dans quelle ville fait-il le plus chaud ? » — **sans autoriser le moindre appel d'outil** (`tools=[]`) : le modèle doit répondre depuis l'historique.

**Indices** :
- `# Indice` : après la réponse de l'assistant, ajouter le message `{"role": "tool", "tool_call_id": ..., "content": ...}` — c'est lui qui met le résultat à disposition des tours suivants
- `# Indice` : `get_weather` est simulée : Lyon 22 °C, Marseille 26 °C (voir la Section 1) — la bonne réponse au tour 3 est donc Marseille
- `# Indice` : au tour 3, passer `tools=[]` : sans schéma déclaré, le modèle ne peut plus structurer aucun appel

In [18]:
# Exercice 2 : Conversation multi-tours -- reutiliser un resultat d'outil au tour suivant
# TODO etudiant : construire un historique messages qui survit aux trois tours

historique_ex2 = [{"role": "user", "content": "Quel temps fait-il a Lyon ?"}]

# Etape 1 : Tour 1 -- appel de get_weather puis reinjection du resultat dans l'historique
# TODO etudiant : response = client.chat.completions.create(..., tools=tools), append de la
#   reponse de l'assistant, execution de get_weather(**args), append du message role="tool"
#   avec le bon tool_call_id

# Etape 2 : Tour 2 -- meme historique, nouvelle question sur Marseille
# TODO etudiant : append {"role": "user", "content": "Et a Marseille ?"} puis rejouer la boucle

# Etape 3 : Tour 3 -- question de synthese SANS outil (tools=[])
# TODO etudiant : append {"role": "user", "content": "Dans quelle ville fait-il le plus chaud ?"},
#   appel final avec tools=[] : la reponse doit venir de l'historique, pas d'un nouvel appel

reponse_tour3 = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


### Exercice 3 : Le surcoût de tokens d'un schéma d'outils

Chaque appel passé avec `tools` paie le **schéma complet de tous les outils** dans le prompt — même si aucun n'est appelé, même avec `tool_choice="none"`. La documentation le dit ; l'exercice le **mesure**.

**Objectif** : poser deux fois la même question avec `tool_choice="none"` — une fois avec `tools=[]`, une fois avec `tools=extended_tools` (trois outils) — lire `response.usage.prompt_tokens` dans chaque cas, puis calculer l'écart total et l'écart moyen **par outil déclaré**.

**Indices** :
- `# Indice` : `response.usage.prompt_tokens` compte les tokens du prompt, schémas d'outils compris
- `# Indice` : garder `tool_choice="none"` pour qu'aucun outil ne soit appelé — on mesure le prompt, pas le comportement
- `# Indice` : l'écart par outil s'obtient en divisant la différence par `len(extended_tools)`

In [19]:
# Exercice 3 : Le surcout de tokens d'un schema d'outils
# TODO etudiant : mesurer le surcout de prompt_tokens du a la declaration des outils

question_ex3 = "Resume en une phrase ce que tu sais de la tour Eiffel."

# Etape 1 : Appel sans aucun outil declare
# TODO etudiant : reponse_sans = client.chat.completions.create(..., tools=[],
#   tool_choice="none") puis lire reponse_sans.usage.prompt_tokens

# Etape 2 : Meme appel avec les trois outils declares
# TODO etudiant : reponse_avec = client.chat.completions.create(..., tools=extended_tools,
#   tool_choice="none") puis lire reponse_avec.usage.prompt_tokens

# Etape 3 : Ecart total et ecart moyen par outil declare
# TODO etudiant : difference des deux mesures, division par len(extended_tools)

surcout_par_outil = None  # TODO etudiant
print("Exercice a completer")

Exercice a completer


### Exercice 4 : Ancré ou halluciné — réponse avec ou sans accès à la base

La Section 7 a défini `COURSE_DB` et son outil `search_courses` — mais le modèle ne connaît cette base **que s'il l'interroge**. Que répond-il si on lui retire l'outil ?

**Objectif** : poser la question « Quels sont les cours de Machine Learning de plus de 20 heures à moins de 300 EUR ? » de deux façons — sans aucun outil déclaré (`tools=[]`), puis avec l'outil `search_courses` (`tool_choice="auto"`) — et confronter les deux réponses à la **vérité terrain** calculée d'abord en Python sur `COURSE_DB` : pour chaque cours cité, dire s'il existe dans la base, s'il fait partie de la vérité terrain, ou s'il est **inventé**.

**Indices** :
- `# Indice` : vérité terrain = `[c for c in COURSE_DB if c["category"] == "ML" and c["duration_hours"] > 20 and c["price"] < 300]` — une seule entrée
- `# Indice` : sans outil, le modèle n'a aucune connaissance de `COURSE_DB` : soit il invente des cours plausibles, soit il avoue ne pas savoir — les deux comportements sont instructifs
- `# Indice` : avec l'outil, passer par `run_safe_conversation(question, search_tool, search_functions)` (Section 7)
- `# Indice` : la comparaison se fait sur les **titres cités** : dans la vérité terrain, présent dans `COURSE_DB` mais hors critères, ou absent de la base

In [20]:
# Exercice 4 : Ancré ou halluciné -- reponse avec ou sans acces a la base
# TODO etudiant : confronter deux reponses (sans outil / avec outil) a la verite terrain

question_ex4 = "Quels sont les cours de Machine Learning de plus de 20 heures a moins de 300 EUR ?"

# Etape 1 : Verite terrain, calculee en Python sur COURSE_DB
verite_terrain = None  # TODO etudiant : liste des titres de COURSE_DB avec category == "ML",
#   duration_hours > 20 et price < 300

# Etape 2 : Reponse SANS outil declare
# TODO etudiant : client.chat.completions.create(..., tools=[]) -- le modele ne connait pas COURSE_DB

# Etape 3 : Reponse avec l'outil search_courses
# TODO etudiant : run_safe_conversation(question_ex4, search_tool, search_functions)

# Etape 4 : Confrontation des deux reponses a la verite terrain
# TODO etudiant : pour chaque reponse, lister les titres cites et les classer : dans la verite
#   terrain, present dans COURSE_DB mais hors criteres, ou absent de la base (= invente)

print("Exercice a completer")

Exercice a completer


## 8. Conclusion et Bonnes Pratiques

### Points Clés

1. **Descriptions précises** : Les descriptions des fonctions guident le modèle. Soyez explicite !
2. **JSON Schema rigoureux** : Définissez bien les types et contraintes des paramètres
3. **Gestion d'erreurs** : Toujours prévoir des fallbacks et retourner des erreurs structurées
4. **Limite d'itérations** : Évitez les boucles infinies avec un max_iterations
5. **Sécurité** : Validez TOUJOURS les arguments avant exécution (injections, chemins dangereux, etc.)

### Cas d'Usage Réels

| Domaine | Exemple d'Application |
|---------|----------------------|
| **E-commerce** | Recherche produits, ajout panier, suivi commande |
| **Productivité** | Gestion calendrier, emails, rappels |
| **Finance** | Consultation soldes, virements, alertes |
| **Data Science** | Requêtes SQL, visualisations, analyses |
| **DevOps** | Déploiements, logs, monitoring |

### Exercices Suggérés

1. **Système de réservation** : Créez des fonctions pour chercher/réserver des restaurants
2. **Calculatrice avancée** : Fonctions mathématiques (factorielle, fibonacci, primalité)
3. **API réelle** : Intégrez une vraie API météo (OpenWeatherMap) au lieu de la simulation
4. **Multi-agents** : Créez plusieurs "agents" avec des sets de fonctions différents

### Prochaine Étape

Dans le notebook suivant (**05_RAG_Modern.ipynb**), nous verrons comment combiner function calling avec le RAG (Retrieval-Augmented Generation) pour créer des assistants qui peuvent chercher dans des bases de connaissances.

***

**Ressources :**
- [OpenAI Function Calling Guide](https://platform.openai.com/docs/guides/function-calling)
- [JSON Schema Documentation](https://json-schema.org/)
- Documentation Notebook 1 (OpenAI Intro)
- Documentation Notebook 3 (Structured Outputs)

In [21]:
# Cellule de validation finale
print("✓ Notebook Function Calling terminé !")
print("✓ Concepts maîtrisés: Tools, tool_choice, boucles agentiques, gestion d'erreurs")
print("✓ Prochaine étape: RAG (Retrieval-Augmented Generation)")

✓ Notebook Function Calling terminé !
✓ Concepts maîtrisés: Tools, tool_choice, boucles agentiques, gestion d'erreurs
✓ Prochaine étape: RAG (Retrieval-Augmented Generation)


### Validation de la progression

Cette cellule confirme la complétion du notebook et résume les acquis :

**Concepts maîtrisés :**
1. **Tools** : Définition de fonctions avec JSON Schema
2. **tool_choice** : Contrôle fin du comportement du modèle (auto, required, none, spécifique)
3. **Boucles agentiques** : Pattern fondamental pour les agents autonomes
4. **Gestion d'erreurs** : Robustesse en production (timeouts, validation, limites)
5. **Orchestration** : Appels groupés en un seul tour et décomposition automatique de tâches complexes

**Prochaine étape :** Le notebook RAG (Retrieval-Augmented Generation) combinera ces techniques avec des bases de connaissances pour créer des assistants capables de raisonner sur des documents.

***

## EXEMPLE CORRIGÉ - Assistant de Planification Multi-Outils

**Solution par Nathan JULOU et Robin VAZ**

### Objectif

Créer un **assistant de planification** qui combine plusieurs outils pour orchestrer une soirée complète.

### Outils implémentés

| Outil | Description | Paramètres |
|-------|-------------|------------|
| `search_events(city, date, category)` | Rechercher des événements culturels | city, date, category (optionnel) |
| `get_travel_time(origin, destination, transport)` | Calculer le temps de trajet | origin, destination, transport (optionnel) |
| `book_table(restaurant, time, guests, special_requests)` | Réserver une table | restaurant, time, guests, special_requests (optionnel) |

### Techniques utilisées

- **JSON Schema** : Définition rigoureuse des paramètres
- **Boucle agentique** : `run_conversation()` orchestre les appels
- **Appels groupés en un seul tour** : Le modèle peut appeler plusieurs outils simultanément

> **Pattern clé** : Le modèle décide automatiquement quels outils appeler et dans quel ordre.


In [22]:
# EXEMPLE CORRIGÉ: Assistant de Planification Multi-Outils
# Solution par Nathan JULOU et Robin VAZ

# 1. Définition des 3 outils

# Tool search_events
search_events_tool = {
    "type": "function",
    "function": {
        "name": "search_events",
        "description": "Rechercher des événements culturels (ex: concerts, spectacles, expos) dans une ville pour une date donnée",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Nom de la ville (ex: Lyon, Paris)"},
                "date": {"type": "string", "description": "Date de l'événement (ex: 'ce soir', 'demain')"},
                "category": {"type": "string", "enum": ["culturel", "concert", "theatre", "exposition"], "description": "Catégorie d'événement"}
            },
            "required": ["city", "date"]
        }
    }
}

# Tool get_travel_time
get_travel_time_tool = {
    "type": "function",
    "function": {
        "name": "get_travel_time",
        "description": "Obtenir le temps de trajet estimé entre deux lieux à Lyon",
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {"type": "string", "description": "Point de départ (ex: 'Part-Dieu', 'Villeurbanne')"},
                "destination": {"type": "string", "description": "Point d'arrivée (ex: 'Croix-Rousse', 'Tête d'Or')"},
                "transport": {"type": "string", "enum": ["transports", "voiture", "velo", "pieds"], "description": "Mode de transport"}
            },
            "required": ["origin", "destination"]
        }
    }
}

# Tool book_table
book_table_tool = {
    "type": "function",
    "function": {
        "name": "book_table",
        "description": ("Réserver une table dans un restaurant partenaire pour un nombre de personnes à une heure donnée. "
                        "Restaurants partenaires : Le Restaurant Italien (italien, 15-25€), "
                        "Bistrot du Coin (français, 20-35€), Sushi Express (japonais, 12-20€)"),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant": {"type": "string", "description": "Nom exact d'un restaurant partenaire"},
                "time": {"type": "string", "description": "Heure de réservation (ex: '19:00', '20:30')"},
                "guests": {"type": "integer", "description": "Nombre de personnes"},
                "special_requests": {"type": "string", "description": "Requêtes spéciales (ex: 'vue', 'calme')"}
            },
            "required": ["restaurant", "time", "guests"]
        }
    }
}

print("3 outils définis: search_events, get_travel_time, book_table")


3 outils définis: search_events, get_travel_time, book_table


In [23]:
# 2. Implémentation des fonctions simulées

def search_events(city: str, date: str, category: str = "culturel") -> str:
    """Simule une recherche d'événements"""
    events_db = {
        "Lyon": [
            {"title": "Nuits de la Peur", "venue": "Théâtre de la Croix-Rousse", "time": "19:00", "price": "15€"},
            {"title": "Concert Jazz", "venue": "Salle 300", "time": "20:30", "price": "25€"},
            {"title": "Expo Impressionnistes", "venue": "Musée des Beaux-Arts", "time": "10:00-18:00", "price": "12€"},
        ],
        "Paris": [
            {"title": "Hamlet", "venue": "Comédie-Française", "time": "19:30", "price": "35€"},
        ]
    }
    
    city_events = events_db.get(city, [])
    return json.dumps({"city": city, "date": date, "events": city_events}, ensure_ascii=False)

def get_travel_time(origin: str, destination: str, transport: str = "transports") -> str:
    """Simule un calcul de temps de trajet"""
    travel_times = {
        ("Part-Dieu", "Croix-Rousse"): {"transports": "12 min", "voiture": "8 min", "velo": "15 min", "pieds": "25 min"},
        ("Part-Dieu", "Villeurbanne"): {"transports": "10 min", "voiture": "12 min", "velo": "18 min", "pieds": "35 min"},
        ("Croix-Rousse", "Villeurbanne"): {"transports": "20 min", "voiture": "15 min", "velo": "22 min", "pieds": "40 min"},
    }
    
    key = (origin, destination)
    time = travel_times.get(key, {}).get(transport, "15 min (estimé)")
    return json.dumps({"origin": origin, "destination": destination, "transport": transport, "time": time}, ensure_ascii=False)

def book_table(restaurant: str, time: str, guests: int, special_requests: str = "") -> str:
    """Simule une réservation de restaurant"""
    restaurants = {
        "Le Restaurant Italien": {"cuisine": "italien", "price_range": "15-25€"},
        "Bistrot du Coin": {"cuisine": "français", "price_range": "20-35€"},
        "Sushi Express": {"cuisine": "japonais", "price_range": "12-20€"},
    }
    
    info = restaurants.get(restaurant, {"cuisine": "inconnu", "price_range": "N/A"})
    return json.dumps({
        "status": "confirmed",
        "restaurant": restaurant,
        "time": time,
        "guests": guests,
        "cuisine": info["cuisine"],
        "price_range": info["price_range"],
        "special_requests": special_requests
    }, ensure_ascii=False)

print("Fonctions simulées implémentées")


Fonctions simulées implémentées


Les fonctions simulées étant définies, le test de l'assistant de planification peut être lancé. Cette cellule orchestre un appel complet au modèle avec tous les outils disponibles pour répondre à une requête utilisateur complexe.

In [24]:
# 3. Test de l'assistant de planification

all_tools = [search_events_tool, get_travel_time_tool, book_table_tool]
all_functions = {
    "search_events": search_events,
    "get_travel_time": get_travel_time,
    "book_table": book_table
}

print("Outils disponibles:", ", ".join(all_functions.keys()))

# Test avec la requête du challenge
test_request = ("Trouve-moi un événement culturel à Lyon ce soir vers 19h, indique le trajet depuis la Part-Dieu, "
                "puis réserve une table pour deux dans un restaurant italien pas cher à 21h")

print(f"\nQuestion: {test_request}")
print("\nAppels de fonctions:")

result = run_conversation(test_request, all_tools, all_functions)

print("\nRésultat final:")
print(result)


Outils disponibles: search_events, get_travel_time, book_table

Question: Trouve-moi un événement culturel à Lyon ce soir vers 19h, indique le trajet depuis la Part-Dieu, puis réserve une table pour deux dans un restaurant italien pas cher à 21h

Appels de fonctions:


  Appel: search_events({'city': 'Lyon', 'date': 'ce soir', 'category': 'culturel'})


  Appel: get_travel_time({'origin': 'Part-Dieu', 'destination': 'Théâtre de la Croix-Rousse', 'transport': 'transports'})
  Appel: book_table({'restaurant': 'Le Restaurant Italien', 'time': '21:00', 'guests': 2})



Résultat final:
Parfait — tout est prêt. Voici le récapitulatif :

Événement
- Nuits de la Peur
- Lieu : Théâtre de la Croix‑Rousse
- Horaire : 19:00 ce soir
- Tarif : 15€

Trajet depuis la Part‑Dieu
- Mode utilisé : transports en commun
- Durée estimée : ~15 minutes
- Conseils : prévois de partir vers 18:40–18:45 pour être large (contrôles, montée en salle, etc.). Si tu veux, je peux te donner l’itinéraire détaillé (métro/bus/tram) pas à pas.

Réservation au restaurant
- Restaurant : Le Restaurant Italien (partenaire)
- Heure : 21:00 ce soir
- Couvert : 2 personnes
- Confirmée — cuisine italienne, fourchette 15–25€ par personne
- Si tu veux ajouter une requête spéciale (table au calme, anniversaire, etc.), dis‑le moi et je la transmets.

Souhaites‑tu :
- Que je te donne l’itinéraire détaillé (Part‑Dieu → Théâtre) ?
- Que je réserve/achète les billets pour le spectacle ?
- Que je prépare l’itinéraire Théâtre → restaurant après le spectacle ?


### Analyse de l'assistant de planification

**Points forts de l'implémentation :**

1. **Tools bien documentés** : Les descriptions guident le modèle dans ses choix
2. **Fonctions simulées réalistes** : Retours JSON structurés avec données cohérentes
3. **Réutilisation de `run_conversation()`** : Pattern agentique standard

**Que lire dans la sortie :**

- **Un outil par besoin** : `search_events` pour l'événement, `get_travel_time` pour le trajet, `book_table` pour la réservation.
- **L'enchaînement** : la destination du trajet n'est pas dans la demande. Elle vient du résultat de `search_events` (le lieu de l'événement). Le modèle réinjecte un résultat d'outil dans l'appel suivant, ce qui impose deux tours au moins.
- **Le choix du restaurant** : aucun outil ne cherche de restaurant. Le modèle le choisit dans la liste des restaurants partenaires que donne la description de `book_table`. La description d'un outil fait partie de ce que le modèle sait.
- **Les paramètres optionnels** : comparez chaque argument de `book_table` à la demande. Si `special_requests` est rempli, la demande n'en contient pas : le modèle l'a inventé, souvent à partir des exemples de la description du paramètre (`'vue'`, `'calme'`). Dans une application réelle, un argument absent de la demande se fait confirmer par l'utilisateur avant une réservation.

> Si la demande omet une donnée obligatoire, par exemple le nombre de personnes, le modèle ne peut pas appeler `book_table` et demande la précision. C'est le comportement attendu, et la raison pour laquelle la demande de test fournit toutes les données requises.

**Pattern d'orchestration** :
```
Question utilisateur → Analyse → Appel tool → Résultat → Synthèse
```

Le modèle décide automatiquement de l'ordre des opérations sans règle explicite.


***

## CHALLENGE BONUS - Agent de Résolution d'Équations Mathématiques

**Points : 0.75 pts**

### Objectif

Créer un **agent mathématique** capable de décomposer et résoudre des expressions complexes en utilisant des outils de calcul.

### Ce que vous avez appris

L'exemple précédent vous a montré:
- **Définition de tools** avec JSON Schema
- **Fonctions simulées** retournant du JSON
- **Boucle agentique** avec `run_conversation()`

### Nouveau défi : Arbre d'opérations

Contrairement à l'assistant de planification (outils indépendants), vous devez créer un **agent qui décompose une expression** en opérations atomiques :

```
Expression: "calcule (3 + 5) * 2**3"
↓
1. add(3, 5) → 8
2. power(2, 3) → 8
3. multiply(8, 8) → 64
```

### Critères de succès

- [ ] Définir 4 outils : `add(a, b)`, `subtract(a, b)`, `multiply(a, b)`, `power(base, exponent)`
- [ ] Implémenter les fonctions (pas de simulation, vrais calculs)
- [ ] Créer une fonction `solve_math_expression(expr: str) -> dict` qui :
  - Passe l'expression au modèle
  - Laisse le modèle appeler les outils dans l'ordre correct
  - Retourne `{"result": int, "steps": [...]}`
- [ ] Tester avec : `"(10 - 3) * 2 + 5**2"`
- [ ] Le modèle doit appeler les outils dans le bon ordre (parenthèses d'abord)

### Contraintes techniques

- Les outils ne prennent que **2 arguments** (opérations binaires)
- Le modèle doit gérer la **précédence des opérateurs**
- Utiliser `run_conversation()` de la Section 3
- Logger chaque appel avec `print()`

### Différence avec l'exemple corrigé

| Exemple corrigé (Planification) | Nouveau challenge (Maths) |
|--------------------------------|---------------------------|
| Outils indépendants | Outils avec **dépendances** |
| Ordre libre | Ordre **imposé** par les maths |
| Simulation de données | **Vrais calculs** |
| 3 outils | 4 outils |

> **Indices :**

- Le modèle doit comprendre qu'il faut résoudre `(10 - 3)` avant de multiplier
- `power(5, 2)` doit être appelé avant l'addition finale
- Le retour JSON peut inclure `"steps": [{"op": "add", "args": [3, 5], "result": 8}, ...]`

***

**Soumission** : PR avec titre "Challenge #2 - [Votre Nom]" et les 4 tools + fonction `solve_math_expression`


In [25]:
# CHALLENGE: Agent mathématique multi-outils
# TODO: Implémentez votre agent de calcul

# 1. Définir les outils (tools)
# math_tools = [...]

# 2. Implémenter les fonctions
# def add(a, b): ...
# def subtract(a, b): ...
# def multiply(a, b): ...
# def power(base, exponent): ...

# 3. Fonction de résolution avec suivi des étapes
# def solve_math_expression(expr: str) -> dict:
#     steps = []
#     ...utiliser run_conversation()...
#     return {"result": final_result, "steps": steps}

# 4. Tester
# result = solve_math_expression("(10 - 3) * 2 + 5**2")
# print(f"Résultat: {result['result']}")
# print(f"Étapes: {result['steps']}")
print("Challenge a completer")


Challenge a completer
